In [12]:
import sys
!{sys.executable} -m pip install requests

Defaulting to user installation because normal site-packages is not writeable


In [13]:
import requests

url = 'https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201-202602/variaveis/4099?localidades=N3[26]&classificacao=2[all]'

r = requests.get(url)


In [14]:

import pandas as pd

data = pd.read_json(url)

In [15]:
resultados = data['resultados'][0]

dfs = []
for r in resultados:
    nome_categoria = list(r['classificacoes'][0]['categoria'].values())[0].strip().lower()
    serie = r['series'][0]['serie']
    s = pd.DataFrame.from_dict(serie, orient='index', columns=[nome_categoria])
    dfs.append(s)

df = pd.concat(dfs, axis=1)
df.index.name = 'periodo'
df = df.reset_index()

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   periodo   58 non-null     str  
 1   total     58 non-null     str  
 2   homens    58 non-null     str  
 3   mulheres  58 non-null     str  
dtypes: str(4)
memory usage: 1.9 KB


In [17]:
for col in ['total', 'homens', 'mulheres']:
    df[col] = df[col].replace('...', '0').astype(float)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   periodo   58 non-null     str    
 1   total     58 non-null     float64
 2   homens    58 non-null     float64
 3   mulheres  58 non-null     float64
dtypes: float64(3), str(1)
memory usage: 1.9 KB


In [18]:
df['ano'] = df['periodo'].str[:4]
df['tri'] = df['periodo'].str[-2:].astype(int)

df['periodo'] = pd.PeriodIndex(df['ano'] + 'Q' + df['tri'].astype(str), freq='Q')


In [19]:
df['periodo'] = df['periodo'].dt.to_timestamp()

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   periodo   58 non-null     datetime64[us]
 1   total     58 non-null     float64       
 2   homens    58 non-null     float64       
 3   mulheres  58 non-null     float64       
 4   ano       58 non-null     str           
 5   tri       58 non-null     int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 2.8 KB


In [21]:
df_filtrado = df.query("total != 0 or mulheres != 0 or homens != 0")

print("Média - Total:", df_filtrado['total'].mean())
print("Média - Mulheres:", df_filtrado['mulheres'].mean())
print("Média - Homens:", df_filtrado['homens'].mean())

Média - Total: 12.354000000000001
Média - Mulheres: 14.505999999999998
Média - Homens: 10.77


In [22]:
df_filtrado = df.query("total != 0 or mulheres != 0 or homens != 0")

print("Mediana - Total:", df_filtrado['total'].median())
print("Mediana - Mulheres:", df_filtrado['mulheres'].median())
print("Mediana - Homens:", df_filtrado['homens'].median())

Mediana - Total: 11.8
Mediana - Mulheres: 14.3
Mediana - Homens: 10.149999999999999


In [23]:
df_filtrado = df.query("total != 0 or mulheres != 0 or homens != 0")

print("Moda - Total:", df_filtrado['total'].mode())
print("Moda - Mulheres:", df_filtrado['mulheres'].mode())
print("Moda - Homens:", df_filtrado['homens'].mode())

Moda - Total: 0     9.2
1    14.2
Name: total, dtype: float64
Moda - Mulheres: 0    10.4
1    11.1
2    11.8
3    12.9
4    18.6
Name: mulheres, dtype: float64
Moda - Homens: 0     6.8
1     7.4
2     8.2
3     8.3
4    10.0
5    10.3
6    11.0
7    12.5
8    17.4
Name: homens, dtype: float64


In [24]:
df_filtrado.set_index('periodo').resample('YE').mean(numeric_only=True)

,total,homens,mulheres,tri
periodo,,,,
2012-12-31,9.125,7.250000,11.750000,2.5
2013-12-31,9.075,7.300000,11.600000,2.5
2014-12-31,8.250,7.175000,9.775000,2.5
2015-12-31,9.950,8.875000,11.450000,2.5
2016-12-31,14.750,13.450000,16.600000,2.5
2017-12-31,17.850,16.450000,19.750000,2.5
2018-12-31,16.900,15.950000,18.200000,2.5
2019-12-31,15.650,13.575000,18.400000,2.5
2020-12-31,14.800,13.400000,16.700000,1.0


In [25]:
print("A variância total é:", df_filtrado['total'].var())
print("A variância mulheres é:", df_filtrado['mulheres'].var())
print("A variância homens é:", df_filtrado['homens'].var())

A variância total é: 11.14906530612245
A variância mulheres é: 11.75975918367347
A variância homens é: 11.33969387755102


In [26]:
print("O desvio padrão total é:", df_filtrado['total'].std())
print("O desvio padrão mulheres é:", df_filtrado['mulheres'].std())
print("O desvio padrão homens é:", df_filtrado['homens'].std())

O desvio padrão total é: 3.339021609112833
O desvio padrão mulheres é: 3.4292505279832604
O desvio padrão homens é: 3.3674461951976338


In [27]:
print("A amplitude total é:", df_filtrado['total'].max() - df_filtrado['total'].min())
print("A amplitude mulheres é:", df_filtrado['mulheres'].max() - df_filtrado['mulheres'].min())
print("A amplitude homens é:", df_filtrado['homens'].max() - df_filtrado['homens'].min())

A amplitude total é: 11.6
A amplitude mulheres é: 12.1
A amplitude homens é: 11.299999999999999


In [28]:
print("A assimetria total é:", df_filtrado['total'].skew())
print("A assimetria mulheres é:", df_filtrado['mulheres'].skew())
print("A assimetria homens é:", df_filtrado['homens'].skew())

A assimetria total é: 0.27706135911413904
A assimetria mulheres é: 0.07059863683724517
A assimetria homens é: 0.4724228033905601


In [29]:
print("A curtose total é:", df_filtrado['total'].kurt())
print("A curtose mulheres é:", df_filtrado['mulheres'].kurt())
print("A curtose homens é:", df_filtrado['homens'].kurt())

A curtose total é: -1.2083806627771387
A curtose mulheres é: -1.270171387536283
A curtose homens é: -0.9643127575781092
